In [1]:
import duckdb
con = duckdb.connect("../data/processed/retail.duckdb", read_only=True)

In [2]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                            column_names                                                            │                                                        column_types                                                         │ temporary │
│ varchar  │ varchar │ varchar  │                                                             varchar[]                                                              │                                                          varchar[]                                                          │  boolean  │
├──────────┼─────────┼──────────┼────────────────────────────────────────────────────

In [3]:
con.sql("SELECT * FROM raw.sales LIMIT 5")

┌───────────────┬───────────┬─────────┬──────────┬──────────┬─────────┬───────┐
│    item_id    │  dept_id  │ cat_id  │ store_id │ state_id │    d    │ units │
│    varchar    │  varchar  │ varchar │ varchar  │ varchar  │ varchar │ int32 │
├───────────────┼───────────┼─────────┼──────────┼──────────┼─────────┼───────┤
│ HOBBIES_1_001 │ HOBBIES_1 │ HOBBIES │ CA_1     │ CA       │ d_1     │     0 │
│ HOBBIES_1_001 │ HOBBIES_1 │ HOBBIES │ CA_1     │ CA       │ d_2     │     0 │
│ HOBBIES_1_001 │ HOBBIES_1 │ HOBBIES │ CA_1     │ CA       │ d_3     │     0 │
│ HOBBIES_1_001 │ HOBBIES_1 │ HOBBIES │ CA_1     │ CA       │ d_4     │     0 │
│ HOBBIES_1_001 │ HOBBIES_1 │ HOBBIES │ CA_1     │ CA       │ d_5     │     0 │
└───────────────┴───────────┴─────────┴──────────┴──────────┴─────────┴───────┘

In [4]:
con.sql("""
    SELECT store_id, COUNT(*) AS rows, COUNT(DISTINCT item_id) AS items
    FROM raw.sales
    GROUP BY store_id
""")

┌──────────┬─────────┬───────┐
│ store_id │  rows   │ items │
│ varchar  │  int64  │ int64 │
├──────────┼─────────┼───────┤
│ WI_1     │ 5918109 │  3049 │
│ CA_1     │ 5918109 │  3049 │
│ TX_1     │ 5918109 │  3049 │
└──────────┴─────────┴───────┘

In [5]:
con.sql("""
    SELECT c.year, SUM(s.units) AS units
    FROM raw.sales s
    JOIN raw.calendar c ON s.d = c.d
    GROUP BY c.year
    ORDER BY c.year
""")

┌───────┬─────────┐
│ year  │  units  │
│ int32 │ int128  │
├───────┼─────────┤
│  2011 │ 2400370 │
│  2012 │ 3195931 │
│  2013 │ 3665204 │
│  2014 │ 3837314 │
│  2015 │ 4033038 │
│  2016 │ 1654720 │
└───────┴─────────┘

In [6]:
con.sql("""
    SELECT item_id, SUM(units) AS units
    FROM raw.sales
    GROUP BY item_id
    ORDER BY units DESC
    LIMIT 5
""")

┌─────────────┬────────┐
│   item_id   │ units  │
│   varchar   │ int128 │
├─────────────┼────────┤
│ FOODS_3_090 │ 259247 │
│ FOODS_3_586 │ 235429 │
│ FOODS_3_252 │ 152325 │
│ FOODS_3_714 │ 130319 │
│ FOODS_3_555 │ 127314 │
└─────────────┴────────┘

In [7]:
con.sql("SELECT AVG((units = 0)::INT) AS zero_share FROM raw.sales")

┌────────────────────┐
│     zero_share     │
│       double       │
├────────────────────┤
│ 0.6774239316421287 │
└────────────────────┘